# MethylTree QC for Alopex

把本 Notebook 放入已完成 Alopex 上游的项目文件夹，启动现有 MethylTree kernel，
从第一格运行即可；不需要填写项目、Pipeline 或参考文件路径。
Notebook 从 kernel 当前工作目录向上查找 `03_results/run_manifest.json`，
并使用该交付绑定的 Alopex 安装、物种配置、FAI、CpG BED 与可选 TSS BED。

Alopex 的受管 Python 负责运行下游 QC、准备 Single-CpG 及独立的 MethylTree 输入；
MethylTree 分析仍在当前 kernel 中执行。不会启动上游比对或重新提取 CpG。
已有有效 Raw/Single 和下游缓存按原生身份复用；原始整数计数、细胞/条码身份、
坐标、双链 fraction 均值以及观测零和缺失的区别由 Alopex 保持。

结果固定写入项目的 `06_downstream/methyltree/`。X 保持 backed，
原来的筛选门槛、绘图和可选谱系分析保留。Reads–nCG 诊断使用 Alopex 记录的
本细胞拆分后、trimming 前 input pairs，不重新扫描 FASTQ。


In [ ]:
category_key = 'clone_id'
nCG_cutoff = 100000
import_nCG_cutoff = 500000
CORES = 4
CELL_BATCH_SIZE = 8
update = False
recompute = False
SAVE_SAMPLE_INFO = True
RUN_MAPPING_QC = True
RUN_READS_NCG = True
RUN_LINEAGE_ANALYSIS = False
RUN_TSS_PROFILE = False


In [ ]:
from pathlib import Path

import cospar as cs
import matplotlib.pyplot as plt
import methyltree
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

cs.settings.set_figure_params()

## 1. Inputs and reference

In [ ]:
import json
import os
import stat
import subprocess

# 只接受完整交付；坏链接、权限/I/O错误或非普通manifest不向父目录回退。
_start = Path.cwd()
for _project in (_start, *_start.parents):
    _manifest = _project / '03_results/run_manifest.json'
    _probe = _manifest
    while True:
        try:
            _probe.lstat()
        except FileNotFoundError:
            _probe = _probe.parent
        else:
            _state = _probe.stat()
            if _probe != _manifest:
                _state = None
            break
    if _state is not None:
        if not stat.S_ISREG(_state.st_mode):
            raise ValueError(f'Alopex manifest must be a regular file: {_manifest}')
        break
else:
    raise FileNotFoundError('未找到完整 Alopex 交付。请将 Notebook 放入已完成上游的项目目录，并从该目录启动 kernel。')
_delivery = json.loads(_manifest.read_text(encoding='utf-8'))
_pipeline = Path(_delivery['pipeline']['source_root']).expanduser().resolve()
_native_python = _pipeline / 'conda/current/notebook/bin/python'
if not _native_python.is_file():
    raise FileNotFoundError(f'找不到该交付绑定的 Alopex 受管 Python: {_native_python}')

# 在 Alopex 自身环境内准备输入，避免当前 MethylTree kernel 的 Python 版本限制。
_prepare = r"""
import os, sys, tempfile
from pathlib import Path
project, pipeline, cutoff, import_cutoff, recompute, tss = sys.argv[1:]
sys.path.insert(0, str(Path(pipeline) / 'core'))
from alopex.downstream import qc, methyltree as adapter
paths = qc.resolve_notebook_paths(project, pipeline)
qc.run_qc_processor(paths, skip_tss=tss != 'True', tss_sample_fraction=1.0,
    recompute_raw_adata=recompute == 'True', recompute_cpg_stats=recompute == 'True',
    recompute_gini=recompute == 'True', recompute_tss=recompute == 'True')
with qc._qc_write_lock(paths.qc_results_dir):
    samples, h5ad = adapter.prepare_methyltree_inputs(paths,
        nCG_cutoff=int(cutoff), import_nCG_cutoff=int(import_cutoff), recompute_single=recompute == 'True')
    metadata = qc.ad.AnnData(obs=samples.set_index('sample', drop=False).rename_axis(None))
    metadata.uns['cpg_matrix'] = str(h5ad)
    metadata.uns['chromosome_qc'] = qc.load_chromosome_qc_frame(paths)
    if tss == 'True':
        metadata.uns['tss_profile'] = qc.load_notebook_tss_frame(paths).astype({'CloneID': object})
    with tempfile.TemporaryDirectory(dir=qc.project_temporary_parent(h5ad), prefix='methyltree-metadata-') as tmp:
        temporary = Path(tmp) / 'sample_info.h5ad'
        metadata.write_h5ad(temporary)
        os.replace(temporary, h5ad.parent / 'sample_info.h5ad')
"""
subprocess.run([str(_native_python), '-c', _prepare, str(_project), str(_pipeline),
                str(nCG_cutoff), str(import_nCG_cutoff), str(recompute), str(RUN_TSS_PROFILE)],
               env={**os.environ, 'PYTHONPATH': str(_pipeline / 'core'), 'PYTHONDONTWRITEBYTECODE': '1'}, check=True)
data_path = str(_project / '06_downstream/methyltree')
_alopex_inputs = sc.read_h5ad(Path(data_path) / 'sample_info.h5ad')
df_sample = _alopex_inputs.obs.reset_index(drop=True)
_alopex_h5ad = Path(_alopex_inputs.uns['cpg_matrix'])
results_dir = Path(data_path) / 'results'
out_dir = data_path
sample_sheet = Path(data_path) / 'sample_sheet.tsv.gz'
results_dir.mkdir(parents=True, exist_ok=True)
print(f'Alopex project: {_project}\nPipeline: {_pipeline}\nMethylTree output: {data_path}')


In [ ]:
df_chrom = _alopex_inputs.uns['chromosome_qc'].copy()
_primary = df_chrom.loc[df_chrom['chromosome_class'].eq('host')]
df_stat_count = _primary.pivot(index='Sample_ID', columns='chrom', values='observed_sites')
df_stat_rate = _primary.pivot(index='Sample_ID', columns='chrom', values='meth_frac')


## 2. Mapping and methylation QC

QC 和细胞身份直接来自 Alopex。`nCG` 使用 `Sample_Unique_CpG_Sites`；
`met_rate` 使用 `Sample_Meth_CpG_Rate% / 100`，保留 Alopex 的生物学分母。
染色体甲基化率缺失保持 NaN，真实零甲基化仍为 0；不重新扫描原始 CpG。


In [ ]:
df_mapping = df_sample[['sample', 'Mapping_efficiency']].rename(columns={'Mapping_efficiency': 'Mapping_all'})


In [ ]:
df_stat = df_sample.copy()


In [ ]:
# CloneID直接来自Alopex sealed manifest。
df_stat['pass_metQC'] = df_stat['nCG'] > nCG_cutoff
df_stat['HQ'] = df_stat['pass_metQC']
qc_summary = df_stat.groupby(category_key).agg(
    cells=('sample', 'size'), passing_cells=('HQ', 'sum'),
    passing_fraction=('HQ', 'mean'), mean_nCG=('nCG', 'mean'),
)
print(f'Total cells passing QC: {df_stat["HQ"].sum()} / {len(df_stat)}')
qc_summary

In [ ]:
df_C_tmp = df_stat.copy()
categories = df_C_tmp[category_key].unique()
n_cols = len(categories)
fig, axs = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4)) 
for i, cat in enumerate(categories):
    ax = axs[i] if n_cols > 1 else axs 
    subset = df_C_tmp[df_C_tmp[category_key] == cat]    
    sns.histplot(
        data=subset,
        x='nCG',
        ax=ax,
        log_scale=True,
        bins=50,
        color=f'C{i}',
    )    
    ax.axvline(nCG_cutoff, color='red', linestyle='-')
    ax.set_xlabel('Unique CpG (met) counts')
    ax.set_title(f'{cat}\nMean: {subset.nCG.mean()/1000:.1f}K')

plt.tight_layout()
plt.show()

In [ ]:
# select cells for visualization if needed 
df_C_tmp=df_stat
fig,axs=plt.subplots(1,1,figsize=(4,4))
ax=sns.histplot(data=df_C_tmp,x='nCG',ax=axs,log_scale=True,bins=50,hue=category_key)
ax.plot([nCG_cutoff,nCG_cutoff],[0,12],'-r')
ax.set_xlabel('Unique CpG (met) counts')
ax.set_title(f'Mean: {df_C_tmp.nCG.mean()/1000:.0f}K')
plt.tight_layout()

In [ ]:
df_sub=df_stat[df_stat['HQ']]  
for x in ['met_rate','lambda_rate','chrM_rate','pUC19_rate','CHG_rate']:
    if x in df_stat.columns:
        if np.sum(~pd.isna(df_sub[x]))>0:
            sns.histplot(data=df_sub,x=x,hue=category_key)
            mean_x=df_sub[x].mean()
            median_x=df_sub[x].median()
            plt.title(f'{x}; Mean: {mean_x:.3f}; Median: {median_x:.3f}')
            plt.show()
        
for (x,y) in [('CHG_rate','CpG_rate'),('lambda_rate','pUC19_rate'),('lambda_rate','CHG_rate'),('lambda_rate','met_rate'),('pUC19_rate','met_rate'),('lambda_nCG','lambda_rate'),('pUC19_nCG','pUC19_rate')]:
    if (x in df_stat.columns) and (y in df_stat.columns):
        if (np.sum(~pd.isna(df_sub[x]))>0) and (np.sum(~pd.isna(df_sub[y]))>0):
            plt.subplots()
            sns.scatterplot(data=df_sub,x=x,y=y,hue=category_key)
            x_median=df_sub[x].median()
            y_median=df_sub[y].median()
            plt.xlabel(x)
            plt.ylabel(y)
            plt.legend(loc='upper left', bbox_to_anchor=(1.05, 1))
            plt.title(f'{x} Median: {x_median:.3f}; {y} Median: {y_median:.3f}')
            plt.show()

## 3. Sample annotations and spike-ins

In [ ]:
if update:
    df_sample_orig = df_sample.dropna(subset=['nCG']).copy() if 'nCG' in df_sample else df_sample.copy()
df_tmp = df_stat.set_index('sample')
df_sample = df_sample.drop_duplicates('sample', keep='last').set_index('sample')
df_sample = df_sample.reindex(df_sample.index.union(df_tmp.index, sort=False))
for col in df_tmp.columns:
    df_sample.loc[df_tmp.index, col] = df_tmp[col]

df_mapping_tmp = df_mapping.filter(['Mapping_all', 'sample']).rename(
    columns={'Mapping_all': 'Mapping_efficiency'},
).drop_duplicates('sample', keep='last').set_index('sample')
if RUN_MAPPING_QC:
    df_sample['Mapping_efficiency'] = df_mapping_tmp['Mapping_efficiency'].reindex(df_sample.index)
elif 'Mapping_efficiency' not in df_sample:
    df_sample['Mapping_efficiency'] = np.nan
df_sample = df_sample.loc[df_sample.index.isin(df_stat['sample'])].reset_index()
df_sample['tissue'] = df_sample['sample'].str.split('_').str[0]
df_sample['source'] = 'met'
df_sample['mapping_valid'] = df_sample['Mapping_efficiency'] > 50
if update:
    df_sample = pd.concat([df_sample_orig, df_sample], ignore_index=True).drop_duplicates('sample', keep='last')

### Lambda/pUC19 fractions

Reuse the chromosome summary. These totals count cytosine observations, not
raw FASTQ reads. They are added before saving the sample sheet and AnnData.

In [ ]:
_coverage = df_chrom.pivot(index='Sample_ID', columns='chrom', values='coverage').fillna(0)
df_all = pd.DataFrame({'chr1_reads_ratio': _coverage.get('chr1', pd.Series(0, index=_coverage.index)) /
                      _coverage.sum(axis=1).replace(0, np.nan)})
df_all.index.name = 'sample'
df_sample = df_sample.set_index('sample')
for col in df_all.columns:
    df_sample.loc[df_all.index, col] = df_all[col]
df_sample = df_sample.reset_index()

In [ ]:
sns.scatterplot(data=df_sample,x='chr1_reads_ratio',y='lambda_pUC19_reads_ratio',hue=category_key)

In [ ]:
if SAVE_SAMPLE_INFO:
    methyltree.metadata.backup_and_save_sample_info(df_sample, data_path, sample_sheet=sample_sheet)
df_sample.groupby(category_key).agg('mean', numeric_only=True)

## 4. Read backed CpG AnnData

Alopex 按 `import_nCG_cutoff` 准备独立的 Single-CpG 子集。
MethylTree 读取 `06_downstream/methyltree/allCpGs.h5ad`，X 保持 backed；
后面的样本注释与保存逻辑继续使用这份独立矩阵。


In [ ]:
adata_cpg_path = _alopex_h5ad


In [ ]:
adata = sc.read_h5ad(adata_cpg_path, backed='r')


In [ ]:
methyltree.metadata.update_sample_info_on_adata(adata, df_sample)

## 5. Optional lineage analysis

Set `RUN_LINEAGE_ANALYSIS=True` for all-CpG similarity, correction and clone
inference. Results are updated in place while X remains backed. Clone labels,
sizes and p-values are synchronized to `df_sample` by sample ID. They are also
saved to the sample sheet when `SAVE_SAMPLE_INFO=True`.

In [ ]:
if RUN_LINEAGE_ANALYSIS:
    methyltree.similarity.compute_similarity_matrix(
        adata, method='correlation', compute_similarity=True,
        cores=CORES, cell_batch_size=CELL_BATCH_SIZE,
    )
    methyltree.similarity.correct_similarity_matrix(
        adata, show_plots=False,
    )
    methyltree.lineage.infer_tree_from_similarity(adata, method='UPGMA')
    methyltree.clone.infer_clones(
        adata, p_value_threshold=0.05, show_plot=False,
    )
    df_sample = methyltree.metadata.update_df_sample_with_adata(
        df_sample, adata, obs_key=['inferred_clone', 'clone_size', 'clone_pvalue'],
    )
    if SAVE_SAMPLE_INFO:
        methyltree.metadata.backup_and_save_sample_info(df_sample, data_path, sample_sheet=sample_sheet)
    methyltree.plotting.plot_lineage_heatmap(
        adata,
        color_key=['inferred_clone', 'met_rate'],
    )

## 6. Save the annotated matrix

A full save to the same backing file retains X in place.

In [ ]:
methyltree.adata.save_adata_to_h5ad(
    adata, adata.filename if adata.isbacked else adata_cpg_path, light_save=False,
)
adata

## 7. Additional diagnostics

`RUN_READS_NCG=False` 可跳过 reads–nCG 诊断。
reads 直接使用 sealed Alopex manifest 中的本细胞 input pairs；不需要 FASTQ 路径。


### Reads vs nCG: slope and correlation

In [ ]:
corr_result = []
if RUN_READS_NCG:
    clone_ids = [item for item in df_sample[category_key].unique() if pd.notna(item)]
    for clone_id in clone_ids:
        sample_num = 30 ## If use all sample, remove sample_num
        sample_list = df_sample[df_sample[category_key]==clone_id]['sample'].to_list()
        corr, df_reads = methyltree.plotting.plot_reads_nCG(
            data_path, sample_list=sample_list, sample_num=sample_num, title=clone_id,
            df_sample=df_sample, read_counts=df_sample[['sample', 'reads']],
        )
        corr_result.append({'clone_id': clone_id,'corr': corr})

In [ ]:
if RUN_READS_NCG and corr_result:
    corr_df = pd.DataFrame(corr_result) 
    print(corr_df)
    plt.figure(figsize=(6, 6))
    bars = plt.bar(corr_df['clone_id'], corr_df['corr'], 
                   color=[f'C{i}' for i in range(len(corr_df))])
    for i, corr in enumerate(corr_df['corr']):
        plt.text(i, corr + 0.01, f'{corr:.4f}', ha='center', va='bottom', fontsize=10)
    plt.title('Slope Values by Clone id', fontsize=12)
    plt.xlabel('Clone id', fontsize=10)
    plt.ylabel('Slope', fontsize=10)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(fontsize=9)
    plt.legend(bars, corr_df['clone_id'], title='Clone IDs',fontsize=9, title_fontsize=10,bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

### Optional TSS profiles and scores

设置 `RUN_TSS_PROFILE=True` 时，使用 Alopex 物种配置对应的 TSS BED 和原生 profile；
本节继续使用原来的 MethylTree score 与可视化，不需要另行填写参考文件路径。


In [ ]:
df_TSS_new = pd.DataFrame(columns=['sample', 'position_coarse', 'meth_frac'])
if RUN_TSS_PROFILE:
    df_TSS_new = _alopex_inputs.uns['tss_profile'].copy().rename(columns={'Sample_ID': 'sample'})


In [ ]:
df_TSS_score = pd.DataFrame(columns=['sample', 'TSS_score'])
if RUN_TSS_PROFILE and not df_TSS_new.empty:
    df_TSS_score = (
        df_TSS_new.groupby('sample')[['position_coarse', 'meth_frac']]
        .apply(methyltree.qc.compute_TSS_score).rename('TSS_score').reset_index()
    )
df_TSS_score.head()

In [ ]:
if RUN_TSS_PROFILE:
    for sample in df_TSS_score['sample'].head(5):
        plt.figure()
        sns.scatterplot(data=df_TSS_new[df_TSS_new['sample'] == sample], x='position_coarse', y='meth_frac')
        plt.title(sample)

if RUN_TSS_PROFILE:
    df_TSS_score[category_key] = (
        df_TSS_score['sample'].map(df_sample.set_index('sample')[category_key])
    )

if RUN_TSS_PROFILE and not df_TSS_score.empty:
    fig, axs = plt.subplots(1, 1, figsize=(6, 4.5))
    sns.histplot(data=df_TSS_score, x='TSS_score', hue=category_key, bins=50)
    plt.tight_layout()

### Chromosome-level consistency

In [ ]:
df_chr_rate_sub = df_stat_rate.filter([
    x for x in df_stat_rate.columns if x.startswith('chr') and x not in ['chrX', 'chrY', 'chrM']
])
adata_chr = sc.AnnData(df_chr_rate_sub.to_numpy(), obs=pd.DataFrame(index=df_chr_rate_sub.index))
methyltree.metadata.update_sample_info_on_adata(adata_chr, df_sample)
adata_chr = adata_chr[adata_chr.obs['HQ'].fillna(False).astype(bool)].copy()

In [ ]:
ko = 0
num_cells = adata_chr.n_obs
if num_cells > 1:
    print(adata_chr.obs_names[ko])
    for j in range(min(7, num_cells - ko - 1)):
        other = j + ko + 1
        sns.scatterplot(
            x=adata_chr.X[ko], y=adata_chr.X[other],
            label=f'{adata_chr.obs_names[other]}; {np.corrcoef(adata_chr.X[ko], adata_chr.X[other])[0, 1]:.2f}',
        )
    plt.legend(loc=[1.05, 0])
    plt.xlabel('Met rate (cell-1)')
    plt.ylabel('Met rate (cell-2)')
else:
    print('Warning: Cell count needs to be greater than 1')

In [ ]:
if adata_chr.n_obs > 1 and adata_chr.n_vars > 1:
    methyltree.similarity.compute_similarity_matrix(
        adata_chr, method='correlation', compute_similarity=True, show_progress=False, cores=CORES,
    )
    X_similarity = adata_chr.obsm['X_similarity']
    off_diag = X_similarity[np.triu_indices_from(X_similarity, k=1)]
    sns.histplot(off_diag)
    plt.title(f'Mean {off_diag.mean():.2f}')
    plt.xlabel('Cell-Cell similarity')

## 8. Close the backing file

The saved H5AD remains available on disk.

In [ ]:
# Release the backed file handle. The completed H5AD remains on disk.
if isinstance(adata, sc.AnnData) and adata.isbacked:
    adata.file.close()